<div style="font-size: 24px; line-height: 1.6;">

# Survivorship Bias: Missing Failures Are Still Evidence

## Missing rows and missing values are clues, not noise

</div>

<div style="font-size: 24px; line-height: 1.6;">

## Takeaway

Functions introduced: `pd.read_json`, `isna`, `isna().sum`, `value_counts`, `pd.crosstab`, `fillna`, `dropna`, `pd.merge`, `duplicated`, `drop_duplicates`.

**Concept learned: missing rows and missing values are evidence.**

</div>

<div style="font-size: 24px; line-height: 1.6;">

### Imports

</div>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)

<div style="font-size: 24px; line-height: 1.6;">

## The story

Revenue exists for surviving startups. Failed startups often have missing revenue, or vanish from the dataset entirely. If you only analyze survivors, you are studying winners and calling it the population.

</div>

<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 1. Load the startup dataset

This file is JSON — common when data comes from a web API. We wrote it in *records* orientation (a list of row-dicts), so we pass `orient='records'` to `pd.read_json()` to read it back.

</div>

In [2]:
startups = pd.read_json("../data/startup_survivorship.json", orient="records")
startups.head()

,startup_id,sector,seed_funding_millions,market_score,survived_3yr,year3_revenue_millions
0,1,Health,0.36,-0.47,True,36.36
1,2,AI,0.70,0.12,True,28.05
2,3,Retail,0.13,0.40,False,NaN
3,4,Games,1.60,1.20,False,NaN
4,5,Health,0.62,0.24,True,97.69


In [3]:
startups.info()

<class 'pandas.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   startup_id              2400 non-null   int64  
 1   sector                  2400 non-null   str    
 2   seed_funding_millions   2400 non-null   float64
 3   market_score            2400 non-null   float64
 4   survived_3yr            2400 non-null   bool   
 5   year3_revenue_millions  634 non-null    float64
dtypes: bool(1), float64(3), int64(1), str(1)
memory usage: 108.5 KB


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 2. Missingness map with `df.isna()`

</div>

In [4]:
startups.isna().head()

,startup_id,sector,seed_funding_millions,market_score,survived_3yr,year3_revenue_millions
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,True
3,False,False,False,False,False,True
4,False,False,False,False,False,False


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 3. Missing counts with `df.isna().sum()`

</div>

In [5]:
startups.isna().sum().sort_values(ascending=False)

year3_revenue_millions    1766
startup_id                   0
sector                       0
seed_funding_millions        0
market_score                 0
survived_3yr                 0
dtype: int64

<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 4. Outcome counts

</div>

In [6]:
startups["survived_3yr"].value_counts()

survived_3yr
False    1766
True      634
Name: count, dtype: int64

<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 5. Cross missingness with the outcome

</div>

In [7]:
pd.crosstab(startups["survived_3yr"], startups["year3_revenue_millions"].isna(),
            rownames=["survived_3yr"], colnames=["revenue_missing"])

revenue_missing,False,True
survived_3yr,,
False,0,1766
True,634,0


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 6. Fill carefully with `df.fillna()`

</div>

In [8]:
startups["year3_revenue_millions"].fillna(0).describe()

count    2400.000000
mean        5.366379
std        12.570705
min         0.000000
25%         0.000000
50%         0.000000
75%         4.605000
max       147.980000
Name: year3_revenue_millions, dtype: float64

<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 7. Drop carefully with `df.dropna()`

</div>

In [9]:
survivors_only = startups.dropna(subset=["year3_revenue_millions"])

<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 8. Compare before and after

</div>

In [10]:
print("Full dataset shape:   ", startups.shape)
print("Survivors-only shape: ", survivors_only.shape)

print("\nSector counts (full):")
print(startups["sector"].value_counts())

print("\nSector counts (survivors only):")
print(survivors_only["sector"].value_counts())

Full dataset shape:    (2400, 6)
Survivors-only shape:  (634, 6)

Sector counts (full):
sector
Retail     613
AI         474
Fintech    469
Games      443
Health     401
Name: count, dtype: int64

Sector counts (survivors only):
sector
Retail     161
AI         142
Games      121
Health     108
Fintech    102
Name: count, dtype: int64


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 9. The biased headline vs. the honest one

</div>

In [11]:
biased = survivors_only["year3_revenue_millions"].mean()
honest_zero_fill = startups["year3_revenue_millions"].fillna(0).mean()
print(f"Survivor-only average revenue: {biased:.2f}M")
print(f"All startups (failed = 0):     {honest_zero_fill:.2f}M")

Survivor-only average revenue: 20.31M
All startups (failed = 0):     5.37M


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 10. Joining tables with `pd.merge()`

Joins can *create* survivorship bias. A press directory only writes about companies that are still alive — merge against it with the default `how='inner'` and every failed startup silently disappears. `how='left'` keeps the full population.

</div>

In [12]:
press = survivors_only[["startup_id"]].copy()
press["press_article"] = "feature story"

inner = startups.merge(press, on="startup_id")
left = startups.merge(press, on="startup_id", how="left")
print("all startups:      ", startups.shape)
print("inner merge result:", inner.shape, " <- failures silently gone")
print("left merge result: ", left.shape)

all startups:       (2400, 6)
inner merge result: (634, 7)  <- failures silently gone
left merge result:  (2400, 7)


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 11. Duplicate rows with `df.duplicated()`

</div>

In [13]:
startups.duplicated().sum()

np.int64(0)

<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 12. Remove duplicates with `df.drop_duplicates()`

</div>

In [14]:
clean = startups.drop_duplicates()
print(startups.shape, clean.shape)

(2400, 6) (2400, 6)


<div style="font-size: 24px; line-height: 1.6;">

## Discussion

- What would the dataset look like if it were scraped from success-story blog posts?
- If we report the average revenue from `survivors_only`, what claim are we implicitly making about the failed startups?

</div>

<div style="font-size: 24px; line-height: 1.6;">

## Don't change data silently

Prefer creating a new object over overwriting the original during EDA — your future self will thank you. That is why we wrote `survivors_only = startups.dropna(...)` above instead of overwriting `startups`.

</div>